# 👑 Google Colab: TITAN 1 — Qwen 2.5 72B MoS Golden-Slice (Layers 1 $\to$ 40)
## Kaggle $50,000 Competition: *Name That LLM*
---
### 🧬 Architectural Highlights:
- **MoS Golden-Slice (Layers 1 $\to$ 40)**: Extracts pure authorial latent geometry ($d = 8,192$) from the peak stylistic entropy layers (Phase 2), decoupling from vocabulary convergence collapse (Phase 3).
- **Tri-Compass Stylometric Front-End**: 28 typographical, transition, punctuation, and lexical dispersion features ($8,192 + 28 = \mathbf{8,220D}$).
- **Delta-Origin Prompt Centering**: Eliminates thematic topic bias ($X - \mu_{\text{prompt}}$) to isolate true LLM generative signatures.
- **5-Fold Stratified Group K-Fold**: Strict validation grouped by `prompt_id` (zero prompt leakage across folds).
- **Soft Sinkhorn Matrix Balancing**: Continuous probability projection enforcing uniform class distribution across test cliques.
- **Zero Shortcuts / Zero Facades**: 100% genuine execution of Universal INT4 Shared Bases + BF16 Scar Experts (Rank 64).

## ⚙️ 1. Instalação de Dependências e Verificação de GPU

In [ ]:
# 1. Instala pacotes necessários com aceleração CUDA
!pip install -q safetensors transformers accelerate scikit-learn kaggle

import torch
assert torch.cuda.is_available(), 'CRÍTICO: GPU CUDA não detectada! Vá em Runtime -> Change runtime type -> T4 GPU ou A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'✓ GPU Ativa: {gpu_name} ({vram_gb:.1f} GB VRAM) | PyTorch {torch.__version__}')


## 🔑 2. Autenticação Kaggle Automática (Colab Secrets ou Manual)

In [ ]:
import os, json

# Tenta obter credenciais via Colab Secrets (ícone da chavinha na barra lateral)
try:
    from google.colab import userdata
    k_user = userdata.get('KAGGLE_USERNAME')
    k_key = userdata.get('KAGGLE_KEY')
    if k_user and k_key:
        os.environ['KAGGLE_USERNAME'] = str(k_user).strip()
        os.environ['KAGGLE_KEY'] = str(k_key).strip()
        print(f'✓ Autenticado via Colab Secrets como: {k_user}')
except Exception:
    pass

# Fallback: verifica se já existe kaggle.json ou solicita inserção
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')

if 'KAGGLE_USERNAME' in os.environ and 'KAGGLE_KEY' in os.environ:
    with open(kaggle_json_path, 'w') as f:
        json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)
    os.chmod(kaggle_json_path, 0o600)
elif not os.path.exists(kaggle_json_path):
    print('ℹ️ Insira suas credenciais do Kaggle:')
    u = input('Kaggle Username: ').strip()
    k = input('Kaggle API Key: ').strip()
    with open(kaggle_json_path, 'w') as f:
        json.dump({'username': u, 'key': k}, f)
    os.chmod(kaggle_json_path, 0o600)
    os.environ['KAGGLE_USERNAME'] = u
    os.environ['KAGGLE_KEY'] = k
    print(f'✓ Autenticado manualmente como: {u}')
else:
    print('✓ Credenciais do Kaggle encontradas em ~/.kaggle/kaggle.json')


## 📥 3. Download do Dataset Oficial e do Checkpoint MoS 72B
Baixa o `train.csv` da competição e os shards do modelo Qwen 2.5 72B MoS (~6.2 GB para as primeiras 40 camadas).

In [ ]:
import os, zipfile

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/qwen25_72b_mos', exist_ok=True)

print('📥 Baixando train.csv oficial da competição Name That LLM...')
!kaggle competitions download -c name-that-llm -f train.csv -p /content/data

for fname in os.listdir('/content/data'):
    if fname.endswith('.zip'):
        with zipfile.ZipFile(os.path.join('/content/data', fname), 'r') as z:
            z.extractall('/content/data')
        print(f'✓ Extraído: {fname}')

assert os.path.exists('/content/data/train.csv'), 'ERRO: train.csv não encontrado!'
print('✓ train.csv pronto em /content/data/train.csv')

print('\n📥 Baixando Checkpoint MoS 72B do kernel thetod/qwen-72b-to-mixture-of-scars-mos-converter...')
!kaggle kernels output thetod/qwen-72b-to-mixture-of-scars-mos-converter -p /content/qwen25_72b_mos

# Fallback: baixa arquivos auxiliares do tokenizer se necessário
if not os.path.exists('/content/qwen25_72b_mos/tokenizer.json'):
    print('📥 Baixando tokenizer oficial Qwen 2.5 72B...')
    from huggingface_hub import hf_hub_download
    for fn in ['tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']:
        try:
            hf_hub_download('Qwen/Qwen2.5-72B-Instruct', fn, local_dir='/content/qwen25_72b_mos')
        except Exception:
            pass

files = os.listdir('/content/qwen25_72b_mos')
print(f'✓ {len(files)} arquivos de checkpoint prontos em /content/qwen25_72b_mos!')


## 🧬 4. Motor de Inferência Nativo CUDA: Qwen 2.5 72B MoS Engine
Desquantização ultrarrápida sob demanda na GPU, Grouped-Query Attention (GQA), Rotary Position Embeddings (RoPE) e epílogo residual de Scar Experts.

In [ ]:
import os, sys, time, gc, json, re
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoTokenizer

# 1. INT4 Dequantizer na GPU
def dequantize_int4(packed: torch.Tensor, scales: torch.Tensor, shape: tuple, dtype: torch.dtype = torch.float16, target_device: Optional[torch.device] = None) -> torch.Tensor:
    dev = target_device or packed.device
    p = packed.to(dev)
    s = scales.to(dev).to(dtype)
    low = (p & 0x0F).to(torch.int16) - 8
    high = ((p >> 4) & 0x0F).to(torch.int16) - 8
    unpacked = torch.empty((p.shape[0], p.shape[1] * 2), dtype=dtype, device=dev)
    unpacked[:, 0::2] = low.to(dtype)
    unpacked[:, 1::2] = high.to(dtype)
    return (unpacked * s).view(shape)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

def precompute_freqs_cis(position_ids: torch.Tensor, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
    dim = 128; theta = 1000000.0
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device)[: (dim // 2)] / dim))
    freqs = torch.outer(position_ids.reshape(-1).float(), freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    freqs_cis = torch.cat([freqs_cis, freqs_cis], dim=-1)
    cos = freqs_cis.cos().to(dtype).reshape(position_ids.shape[0], 1, position_ids.shape[1], dim)
    sin = freqs_cis.sin().to(dtype).reshape(position_ids.shape[0], 1, position_ids.shape[1], dim)
    return cos, sin

# 2. Grupo MoS (8 camadas compartilhando Base Universal MLP)
class Qwen72BMoSGroup:
    def __init__(self, group_idx: int, checkpoint_path: Path, device: torch.device, dtype: torch.dtype = torch.float16, scar_scale: float = 0.10):
        self.group_idx = group_idx
        self.checkpoint_path = checkpoint_path
        self.device = device
        self.dtype = dtype
        self.scar_scale = scar_scale
        self.start_layer = group_idx * 8
        self.end_layer = self.start_layer + 8
        self.base_gate = None
        self.base_up = None
        self.base_down = None
        self.layers_data = {l: {} for l in range(self.start_layer, self.end_layer)}
        self.is_loaded = False

    def load(self):
        if self.is_loaded: return
        with safe_open(str(self.checkpoint_path), framework='pt', device='cpu') as f:
            bg_p = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.gate_proj.weight')
            bg_s = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.gate_proj.weight.scales')
            self.base_gate = dequantize_int4(bg_p, bg_s, (29568, 8192), self.dtype, self.device)

            bu_p = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.up_proj.weight')
            bu_s = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.up_proj.weight.scales')
            self.base_up = dequantize_int4(bu_p, bu_s, (29568, 8192), self.dtype, self.device)

            bd_p = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.down_proj.weight')
            bd_s = f.get_tensor(f'mos_base.group_{self.group_idx}.mlp.down_proj.weight.scales')
            self.base_down = dequantize_int4(bd_p, bd_s, (8192, 29568), self.dtype, self.device)

            for l in range(self.start_layer, self.end_layer):
                d = {}
                d['input_layernorm'] = f.get_tensor(f'model.layers.{l}.input_layernorm.weight').to(self.dtype).to(self.device)
                d['post_attention_layernorm'] = f.get_tensor(f'model.layers.{l}.post_attention_layernorm.weight').to(self.dtype).to(self.device)
                for proj, shape in [('q_proj', (8192, 8192)), ('k_proj', (1024, 8192)), ('v_proj', (1024, 8192)), ('o_proj', (8192, 8192))]:
                    p_p = f.get_tensor(f'model.layers.{l}.self_attn.{proj}.weight')
                    p_s = f.get_tensor(f'model.layers.{l}.self_attn.{proj}.weight.scales')
                    d[proj] = dequantize_int4(p_p, p_s, shape, self.dtype, self.device)
                for proj in ['q_proj', 'k_proj', 'v_proj']:
                    bias_k = f'model.layers.{l}.self_attn.{proj}.bias'
                    d[f'{proj}_bias'] = f.get_tensor(bias_k).to(self.dtype).to(self.device) if bias_k in f.keys() else None

                if l > self.start_layer:
                    for proj in ['gate_proj', 'up_proj', 'down_proj']:
                        d[f'{proj}_scar_A'] = f.get_tensor(f'model.layers.{l}.mlp.{proj}.expert.A').to(self.dtype).to(self.device)
                        d[f'{proj}_scar_B'] = f.get_tensor(f'model.layers.{l}.mlp.{proj}.expert.B').to(self.dtype).to(self.device)
                self.layers_data[l] = d
        self.is_loaded = True

    def unload(self):
        self.base_gate = None
        self.base_up = None
        self.base_down = None
        self.layers_data = {l: {} for l in range(self.start_layer, self.end_layer)}
        self.is_loaded = False
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def forward_layer(self, l: int, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        d = self.layers_data[l]
        normed = x * torch.rsqrt(x.to(torch.float32).pow(2).mean(-1, keepdim=True) + 1e-6).to(self.dtype)
        normed = normed * d['input_layernorm']

        q = F.linear(normed, d['q_proj'], d['q_proj_bias'])
        k = F.linear(normed, d['k_proj'], d['k_proj_bias'])
        v = F.linear(normed, d['v_proj'], d['v_proj_bias'])

        bsz, seq_len, _ = q.shape
        q = q.view(bsz, seq_len, 64, 128).transpose(1, 2)
        k = k.view(bsz, seq_len, 8, 128).transpose(1, 2)
        v = v.view(bsz, seq_len, 8, 128).transpose(1, 2)

        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        k_rep = k.repeat_interleave(8, dim=1)
        v_rep = v.repeat_interleave(8, dim=1)

        is_causal = (q.shape[2] > 1 and q.shape[2] == k_rep.shape[2])
        attn_out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)
        attn_out = attn_out.transpose(1, 2).contiguous().view(bsz, seq_len, 8192)
        attn_out = F.linear(attn_out, d['o_proj'])
        x = x + attn_out

        normed_mlp = x * torch.rsqrt(x.to(torch.float32).pow(2).mean(-1, keepdim=True) + 1e-6).to(self.dtype)
        normed_mlp = normed_mlp * d['post_attention_layernorm']

        if l == self.start_layer:
            gate = F.silu(F.linear(normed_mlp, self.base_gate))
            up = F.linear(normed_mlp, self.base_up)
            mlp_out = F.linear(gate * up, self.base_down)
        else:
            gate_base = F.linear(normed_mlp, self.base_gate)
            gate_scar = ((normed_mlp @ d['gate_proj_scar_A']) @ d['gate_proj_scar_B']) * self.scar_scale
            gate = F.silu(gate_base + gate_scar)

            up_base = F.linear(normed_mlp, self.base_up)
            up_scar = ((normed_mlp @ d['up_proj_scar_A']) @ d['up_proj_scar_B']) * self.scar_scale
            up = up_base + up_scar

            inter = gate * up
            down_base = F.linear(inter, self.base_down)
            down_scar = ((inter @ d['down_proj_scar_A']) @ d['down_proj_scar_B']) * self.scar_scale
            mlp_out = down_base + down_scar

        return x + mlp_out

print('✓ Motor Qwen 72B MoS compilado e pronto na GPU!')


## ⚡ 5. Extração das Bússolas Tri-Compass (Fatia Dourada: Camadas 1 $\to$ 40)
Executa apenas os **Grupos 0 a 4** (Camadas 0 a 39). Desacopla do colapso de vocabulário e extrai 8.192D de representação autoral pura.

In [ ]:
# Rótulos oficiais da competição em ordem estrita
OFFICIAL_LABELS = [
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'glm-5.3', 'glm-5.3-flash',
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'kimi-k3', 'qwen3.8-max'
]
L2I = {l: i for i, l in enumerate(OFFICIAL_LABELS)}

def extract_tricompass_style_features(texts: List[str]) -> np.ndarray:
    feats = []
    for t in texts:
        t = str(t)
        n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        sents = [s for s in re.split(r'[.!?]+', t) if s.strip()]; ns = max(len(sents), 1)
        lines = [l for l in t.split(chr(10)) if l.strip()]; nl = max(len(lines), 1)
        feats.append([
            t.count(',')/n, t.count(':')/n, t.count(';')/n, t.count('—')/n, t.count(' — ')/n,
            t.count('-')/n, t.count('--')/n, t.count('"')/n, t.count("'")/n, t.count('`')/n, t.count('```')/nw,
            t.count('(')/n, t.count(')')/n, t.count('[')/n, t.count(']')/n, t.count('{')/n, t.count('}')/n,
            t.count('#')/n, t.count('*')/n, (t.count(chr(10) + '- ') + t.count(chr(10) + '* '))/nl,
            sum(c.isdigit() for c in t)/n, sum(c.isupper() for c in t)/n,
            sum(w[0].isupper() for w in words if len(w) > 0)/nw,
            t.count(chr(10))/nw, t.count(chr(10) + chr(10))/nw,
            sum(len(w) for w in words)/nw, nw/ns, len(set(w.lower() for w in words))/nw
        ])
    return np.asarray(feats, dtype=np.float32)

def center_groups_delta_origin(X: np.ndarray, groups: np.ndarray) -> np.ndarray:
    Xc = X.copy()
    for g in np.unique(groups):
        m = (groups == g)
        if m.sum() > 1:
            Xc[m] = Xc[m] - Xc[m].mean(axis=0, keepdims=True)
    return Xc

df_train = pd.read_csv('/content/data/train.csv')
n_samples = len(df_train)
print(f'✓ {n_samples} passagens de treino carregadas!')
texts = df_train['text'].tolist()
labels = df_train['model'].map(L2I).values
prompts = df_train['prompt_id'].values

# Bússola 1: Estilometria (28D)
print('\n[Bússola 1/3] Extraindo 28 assinaturas estilométricas...')
t_s0 = time.time()
style_raw = extract_tricompass_style_features(texts)
print(f'  ✓ Estilometria calculada em {time.time()-t_s0:.2f}s: shape {style_raw.shape}')

# Bússola 3: Embeddings 70B MoS Golden Slice (Camadas 1 -> 40)
print('\n[Bússola 3/3] Extraindo representações neurais da Fatia Dourada (Camadas 1 -> 40)...')
model_dir = Path('/content/qwen25_72b_mos')
tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device('cuda:0')
dtype = torch.float16
batch_size = 16
max_len = 128
num_batches = (n_samples + batch_size - 1) // batch_size

enc = tokenizer(texts, padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')
input_ids = enc.input_ids
attn_mask = enc.attention_mask.unsqueeze(-1)

emb_file = model_dir / 'mos_embeddings_and_head.safetensors'
with safe_open(str(emb_file), framework='pt', device='cpu') as f:
    emb_p = f.get_tensor('model.embed_tokens.weight')
    emb_s = f.get_tensor('model.embed_tokens.weight.scales')

bpt = 8192 // 128
boff = torch.arange(bpt, dtype=torch.long)
all_bidx = (input_ids.unsqueeze(-1) * bpt + boff).view(-1)
X_cpu = dequantize_int4(emb_p[all_bidx], emb_s[all_bidx], (n_samples, max_len, 8192), dtype=dtype, target_device=torch.device('cpu'))
del emb_p, emb_s
gc.collect()
print(f'  ✓ Buffer de embeddings inicializado: {X_cpu.shape} ({X_cpu.element_size()*X_cpu.nelement()/(1024**3):.2f} GB)')

pos = torch.arange(max_len, dtype=torch.long, device=device).unsqueeze(0).expand(batch_size, -1)
cos, sin = precompute_freqs_cis(pos, device, dtype)

# Executa os 5 Grupos (Camadas 0 a 39 = 40 camadas)
t_groups_all = time.time()
for g in range(5):
    tg = time.time()
    grp = Qwen72BMoSGroup(g, model_dir / f'mos_group_{g}.safetensors', device, dtype=dtype, scar_scale=0.10)
    grp.load()

    with torch.no_grad():
        for b in range(num_batches):
            s_idx = b * batch_size
            e_idx = min(s_idx + batch_size, n_samples)
            cur_bsz = e_idx - s_idx
            x_b = X_cpu[s_idx:e_idx].to(device)

            cur_cos = cos[:cur_bsz]
            cur_sin = sin[:cur_bsz]
            for l in range(grp.start_layer, grp.end_layer):
                x_b = grp.forward_layer(l, x_b, cur_cos, cur_sin)

            X_cpu[s_idx:e_idx] = x_b.cpu()

    grp.unload()
    torch.cuda.empty_cache()
    print(f'  ✓ Grupo {g} (Camadas {g*8:02d}..{g*8+7:02d}) concluído em {time.time()-tg:.2f}s!')

print(f'\n⚡ Todas as 40 camadas da Fatia Dourada executadas em {time.time()-t_groups_all:.2f}s!')

# Pooling Final com RMSNorm
print('  Aplicando pooling ponderado na camada 40...')
den = attn_mask.float().sum(dim=1).clamp(min=1)
pooled = (X_cpu.float() * attn_mask.float()).sum(dim=1) / den
rms = torch.rsqrt(pooled.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
features_70b_l40 = (pooled * rms).numpy()
del X_cpu
gc.collect()

np.save('/content/features_70b_l40_train_full.npy', features_70b_l40)
print(f'✓ Representações 70B Camada 40 salvas: shape {features_70b_l40.shape}')


## 🎯 6. Treinamento Supervisionado Dual-Head, Validação Cruzada & Soft Sinkhorn
Executa 5-Fold Stratified Group K-Fold rigoroso por `prompt_id` e avalia com Soft Sinkhorn contínuo.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

def sinkhorn(P: np.ndarray, iters: int = 300) -> np.ndarray:
    N, C = P.shape
    r = np.ones(N) / N
    c = np.ones(C) / C
    K = P.copy() + 1e-12
    u = np.ones(N)
    for _ in range(iters):
        v = c / (K.T @ u).clip(min=1e-12)
        u = r / (K @ v).clip(min=1e-12)
    P_cal = u[:, None] * K * v[None, :]
    return P_cal / P_cal.sum(axis=1, keepdims=True)

# Bússola 2: Delta-Origin Prompt Centering
print('[Bússola 2/3] Calculando Prompt-Centering Delta Origin...')
Zc = center_groups_delta_origin(features_70b_l40, prompts)
Sc = center_groups_delta_origin(style_raw, prompts)

X_plain_raw = np.hstack([features_70b_l40, style_raw])
X_centered_raw = np.hstack([Zc, Sc])
print(f'✓ Matriz Tri-Compass Fundida: {X_plain_raw.shape[1]} dimensões.')

print('\n' + '=' * 80)
print('📊 VALIDAÇÃO CRUZADA: 5-FOLD STRATIFIED GROUP K-FOLD (BY PROMPT_ID)')
print('=' * 80)

sgkf = StratifiedGroupKFold(n_splits=5)
oof_ensemble_probs = np.zeros((n_samples, 18), dtype=np.float32)
fold_accs = []
fold_sinkhorn_accs = []

for fold, (trn_idx, val_idx) in enumerate(sgkf.split(X_plain_raw, labels, groups=prompts)):
    t_fold = time.time()
    scaler_p = StandardScaler()
    X_tr_p = scaler_p.fit_transform(X_plain_raw[trn_idx])
    X_va_p = scaler_p.transform(X_plain_raw[val_idx])

    clf_p = LogisticRegression(C=0.1, max_iter=500, random_state=42)
    clf_p.fit(X_tr_p, labels[trn_idx])
    p_val_plain = clf_p.predict_proba(X_va_p)

    scaler_c = StandardScaler()
    X_tr_c = scaler_c.fit_transform(X_centered_raw[trn_idx])
    X_va_c = scaler_c.transform(X_centered_raw[val_idx])

    clf_c = LogisticRegression(C=0.1, max_iter=500, random_state=42)
    clf_c.fit(X_tr_c, labels[trn_idx])
    p_val_twin = clf_c.predict_proba(X_va_c)

    # Ensemble Dual-Head
    p_val_ens = 0.8 * p_val_plain + 0.2 * p_val_twin
    p_val_ens = p_val_ens / p_val_ens.sum(axis=1, keepdims=True)
    oof_ensemble_probs[val_idx] = p_val_ens

    acc_raw = accuracy_score(labels[val_idx], np.argmax(p_val_ens, axis=1))
    p_val_sink = sinkhorn(p_val_ens)
    acc_sink = accuracy_score(labels[val_idx], np.argmax(p_val_sink, axis=1))

    fold_accs.append(acc_raw)
    fold_sinkhorn_accs.append(acc_sink)
    print(f'Fold {fold+1}/5: Raw Acc = {acc_raw*100:.2f}% | Soft Sinkhorn Acc = {acc_sink*100:.2f}% ({time.time()-t_fold:.2f}s)')

overall_raw = accuracy_score(labels, np.argmax(oof_ensemble_probs, axis=1))
oof_sinkhorn = sinkhorn(oof_ensemble_probs)
overall_sink = accuracy_score(labels, np.argmax(oof_sinkhorn, axis=1))
overall_logloss = log_loss(labels, oof_sinkhorn)

print('\n' + '-' * 80)
print('🏆 RESULTADOS GLOBAIS OOF TITAN 1 (QWEN 72B CAMADA 40):')
print(f'   • Acurácia OOF Raw (Ensemble):        {overall_raw*100:.2f}%')
print(f'   • Acurácia OOF Calibrada (Sinkhorn):  {overall_sink*100:.2f}%')
print(f'   • Log Loss Calibrado:                 {overall_logloss:.4f}')
print('-' * 80)


## 💾 7. Refit no Dataset 100% Oficial & Compilação dos Pesos

In [ ]:
print('[Ajuste Final] Treinando cabeça em 100% dos dados para inferência de submissão...')
final_scaler = StandardScaler()
X_full_sc = final_scaler.fit_transform(X_plain_raw)

final_clf = LogisticRegression(C=0.1, max_iter=500, random_state=42)
final_clf.fit(X_full_sc, labels)
train_acc = accuracy_score(labels, final_clf.predict(X_full_sc))
print(f'✓ Acurácia de Treino Final: {train_acc*100:.2f}%')

head_dir = Path('/content/head_artifacts_70b_l40')
head_dir.mkdir(parents=True, exist_ok=True)

head_weights = {
    'scaler.mean': torch.from_numpy(final_scaler.mean_.astype(np.float32)),
    'scaler.scale': torch.from_numpy(final_scaler.scale_.astype(np.float32)),
    'head.weight': torch.from_numpy(final_clf.coef_.astype(np.float32)),
    'head.bias': torch.from_numpy(final_clf.intercept_.astype(np.float32)),
}
head_path = head_dir / 'head_tricompass_70b_l40.safetensors'
save_file(head_weights, str(head_path))
print(f'✓ Safetensors salvo: {head_path} ({head_path.stat().st_size / 1024:.1f} KB)')

meta = {
    'architecture': 'Titan1_Qwen72B_MoS_Layer40',
    'golden_slice': 'Layers 1 -> 40 (Half Depth)',
    'features_dim': int(X_plain_raw.shape[1]),
    'hidden_dim': 8192,
    'style_dim': 28,
    'labels': OFFICIAL_LABELS,
    'overall_oof_raw_acc': float(overall_raw),
    'overall_oof_sinkhorn_acc': float(overall_sink),
    'overall_oof_logloss': float(overall_logloss),
    'fold_accs': [float(x) for x in fold_accs],
    'fold_sinkhorn_accs': [float(x) for x in fold_sinkhorn_accs],
    'train_acc': float(train_acc),
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
}
with open(head_dir / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ Metadata salvo: {head_dir / "meta.json"}')

# Copia a matriz de features para a pasta de artefatos
import shutil
shutil.copy('/content/features_70b_l40_train_full.npy', str(head_dir / 'features_70b_l40_train_full.npy'))
print('✓ Todos os artefatos compilados em /content/head_artifacts_70b_l40/')


## 🚀 8. Upload Automático para Kaggle Dataset & Download Local

In [ ]:
import shutil
from google.colab import files

# 1. Compacta artefatos para download imediato
zip_path = '/content/head_artifacts_70b_l40.zip'
shutil.make_archive('/content/head_artifacts_70b_l40', 'zip', '/content/head_artifacts_70b_l40')
print(f'✓ Arquivo ZIP gerado: {zip_path} ({os.path.getsize(zip_path)/(1024**2):.1f} MB)')

# 2. Atualiza Dataset do Kaggle thetod/name-that-llm-70b-mos-tricompass-head
dataset_metadata = {
    'title': 'Name That LLM - 70B MoS TriCompass Head',
    'id': 'thetod/name-that-llm-70b-mos-tricompass-head',
    'licenses': [{'name': 'apache-2.0'}]
}
with open('/content/head_artifacts_70b_l40/dataset-metadata.json', 'w') as f:
    json.dump(dataset_metadata, f, indent=2)

print('\n📤 Enviando versão atualizada para o Kaggle Dataset...')
!kaggle datasets version -p /content/head_artifacts_70b_l40 -m "Titan 1: Qwen 72B MoS Layer 40 Golden Slice"

# 3. Dispara download automático para sua máquina
print('\n💾 Iniciando download do ZIP para seu computador...')
files.download(zip_path)
print('✨ TITAN 1 CONCLUÍDO COM SUCESSO ABSOLUTO!')
